# TinyVAD vs Energy VAD Evaluation Notebook

This notebook runs a coverage-gated evaluation workflow for TinyVAD and an energy-based VAD baseline. It audits label coverage first, computes metrics only when coverage requirements are met, and then inspects summary outputs and qualitative evidence.

## Workflow Overview

1. Set a stable working directory for reproducible execution.
2. Run split and PHN-label coverage audit.
3. Apply a gate before making ground-truth metric claims.
4. Review comparison and tuning summaries from existing outputs.
5. Inspect qualitative plots for behavior-level differences.
6. Optionally run model-selection and grouped error analysis.
7. Report negative findings and limitations explicitly.

In [1]:
from pathlib import Path
import os

cwd = Path.cwd()
repo_notebook = cwd / 'milestones' / 'personal_vad' / 'notebook'

if repo_notebook.exists():
    os.chdir(repo_notebook)
elif cwd.name == 'notebook' and cwd.parent.name == 'personal_vad':
    pass
else:
    raise FileNotFoundError(f"Cannot resolve notebook working directory from: {cwd}")

print(f'Notebook working directory: {Path.cwd()}')


Notebook working directory: /Users/harrydai/Desktop/HD/CS 6140/project/group/vad-distillation/milestones/personal_vad/notebook


## Split and Label Coverage Audit

Ground-truth metric claims are conditional on coverage checks.

Default gate:
- minimum labeled utterances: 30
- minimum coverage: 60%

Split discipline:
- validation split for selection
- test split for final reporting
- no test-driven tuning

In [2]:
%%bash
../../../.venv/bin/python ../../../milestones/personal_vad/scripts/evaluate_against_ground_truth.py \
  --fold F01 \
  --eval-split test \
  --max-utterances 30 \
  --output-dir ../../../outputs/personal_vad/ground_truth_eval

Processed 10/30 utterances
Processed 20/30 utterances
Processed 30/30 utterances
Coverage audit and 

evaluation complete.
Wrote: ../../../outputs/personal_vad/ground_truth_eval/coverage_report.csv
Wrot

e: ../../../outputs/personal_vad/ground_truth_eval/split_coverage_summary.json
Wrote: ../../../outpu

ts/personal_vad/ground_truth_eval/results.csv
Wrote: ../../../outputs/personal_vad/ground_truth_eval

/summary.json
Wrote: ../../../outputs/personal_vad/ground_truth_eval/summary.txt


In [3]:
%%bash
cat ../../../outputs/personal_vad/ground_truth_eval/split_coverage_summary.json

{
  "timestamp": "2026-03-25T16:27:12",
  "fold_id": "F01",
  "eval_split": "test",
  "gate": {
    

"min_labeled_utterances": 30,
    "min_coverage": 0.6,
    "passed": true,
    "reasons": []
  },
  

"split_stats": {
    "val": {
      "total_utterances": 577,
      "manifest_utterances": 577,
     

 "labeled_utterances": 538,
      "coverage": 0.9324090121317158
    },
    "test": {
      "total_u

tterances": 134,
      "manifest_utterances": 134,
      "labeled_utterances": 134,
      "coverage"

: 1.0
    }
  },
  "label_source": {
    "type": "PHN files under session-level phn_* directories",


    "silence_phones": [
      "sil",
      "noi",
      "noio",
      "sp",
      "spn",
      "pau"


    ],
    "notes": "PHN labels are used as best-available reference and may not perfectly match VA

D boundaries."
  }
}

In [4]:
%%bash
if [ -f ../../../outputs/personal_vad/ground_truth_eval/ground_truth_claims_blocked.txt ]; then
  echo '[Gate blocked] ground-truth metric claims are disabled.'
  cat ../../../outputs/personal_vad/ground_truth_eval/ground_truth_claims_blocked.txt
else
  echo '[Gate passed] showing key ground-truth metrics.'
  ../../../.venv/bin/python - <<'PYIN'
import json
from pathlib import Path

s = json.loads(Path('../../../outputs/personal_vad/ground_truth_eval/summary.json').read_text())
print(f"fold={s['fold_id']} | split={s['eval_split']} | gate_passed={s['gate']['passed']}")
print(f"utterances_evaluated={s['evaluation']['utterances_evaluated']}")
for method, m in s['evaluation']['methods'].items():
    print(
        f"{method}: f1={m['f1']:.4f}, precision={m['precision']:.4f}, "
        f"recall={m['recall']:.4f}, accuracy={m['accuracy']:.4f}, "
        f"segment_iou={m['segment_iou_mean']:.4f}"
    )
PYIN
fi


[Gate passed] showing key ground-truth metrics.


fold=F01 | split=test | gate_passed=True
utterances_evaluated=30
student: f1=0.6613, precision=0.566

0, recall=0.7952, accuracy=0.6103, segment_iou=0.4332
energy_default: f1=0.2810, precision=0.4876, r

ecall=0.1974, accuracy=0.5168, segment_iou=0.2022


## Coverage Gate Block Example

This run demonstrates behavior when label coverage is insufficient. Coverage artifacts are still written, but performance claims are blocked.

In [5]:
%%bash
../../../.venv/bin/python ../../../milestones/personal_vad/scripts/evaluate_against_ground_truth.py \
  --fold FC02 \
  --eval-split test \
  --max-utterances 20 \
  --output-dir ../../../outputs/personal_vad/ground_truth_eval_fc02

cat ../../../outputs/personal_vad/ground_truth_eval_fc02/ground_truth_claims_blocked.txt

Coverage audit complete.
Gate passed: False
Wrote: ../../../outputs/personal_vad/ground_truth_eval_f

c02/coverage_report.csv
Wrote: ../../../outputs/personal_vad/ground_truth_eval_fc02/split_coverage_s

ummary.json
Wrote: ../../../outputs/personal_vad/ground_truth_eval_fc02/ground_truth_claims_blocked.

txt


GROUND-TRUTH METRIC CLAIMS BLOCKED

Cov

erage audit completed, but metric claims are disabled by gate.
Evaluated split: test
Labeled utteran

ces: 0
Coverage: 0.00%
Gate requirements: min_labeled=30, min_coverage=60%

Blocking reasons:
- labe

led_utterances=0 < min_labeled_utterances=30
- coverage=0.000 < min_coverage=0.600

No ground-truth 

performance claim is made in this run.
Use split_coverage_summary.json and coverage_report.csv as ev

idence artifacts.


## Comparison Sanity Check

Quick check from saved comparison JSON (best vs worst agreement).

In [6]:
%%bash
../../../.venv/bin/python - <<'PY'
import json
from pathlib import Path

path = Path('../../../outputs/personal_vad/comparison_live_ready/summary.json')
data = json.loads(path.read_text())
rows = data['comparison']['utterance_comparisons']
best = max(rows, key=lambda x: x['correlation'])
worst = min(rows, key=lambda x: x['correlation'])
print('Comparison sanity check')
print(f'Best agreement: {best["utt_id"]} | corr = {best["correlation"]:.3f} | mse = {best["mse"]:.4f}')
print(f'Worst agreement: {worst["utt_id"]} | corr = {worst["correlation"]:.3f} | mse = {worst["mse"]:.4f}')
PY

Comparison sanity check
Best agreement: F01_Session1_0002 | corr = 0.245 | mse = 0.4213
Worst agreem

ent: F01_Session1_0005 | corr = -0.032 | mse = 0.2649


## Existing Output Summaries

Display the current comparison and tuning summaries.

In [7]:
%%bash
../../../.venv/bin/python - <<'PYIN'
import json
from pathlib import Path

comparison = json.loads(Path('../../../outputs/personal_vad/comparison_live_ready/summary.json').read_text())
tuning = json.loads(Path('../../../outputs/personal_vad/energy_tuning_live_ready/results.json').read_text())

rows = comparison['comparison']['utterance_comparisons']
mean_corr = sum(r['correlation'] for r in rows) / len(rows)
mean_mse = sum(r['mse'] for r in rows) / len(rows)
ckpt = comparison.get('checkpoint_metrics', {})
best = tuning['best_setting']
best_m = tuning['best_metrics']

print('Comparison summary')
print(f"  utterances processed: {comparison['inference']['processed']}")
print(f"  failures: {comparison['inference']['failed']}")
print(f"  model size (MB): {ckpt.get('model_size_mb')}")
print(f"  parameters: {ckpt.get('num_parameters')}")
print(f"  mean correlation: {mean_corr:.4f}")
print(f"  mean mse: {mean_mse:.6f}")

print('')
print('Energy tuning summary')
print(f"  settings evaluated: {tuning['num_settings_evaluated']}")
print(
    f"  best setting: threshold={best['threshold']}, "
    f"h_high={best['hysteresis_high']}, h_low={best['hysteresis_low']}, "
    f"smooth={best['smoothing_window']}"
)
print(f"  best avg correlation: {best_m['avg_correlation']:.4f}")
print(f"  best avg mse: {best_m['avg_mse']:.6f}")
PYIN


Comparison summary
  utterances processed: 5
  failures: 0
  model size (MB): 0.4616584777832031
  p

arameters: 120933
  mean correlation: 0.1008
  mean mse: 0.392055

Energy tuning summary
  settings 

evaluated: 90
  best setting: threshold=0.3, h_high=0.5, h_low=0.2, smooth=3
  best avg correlation:

 0.3684
  best avg mse: 0.393847


## Qualitative Output

![Qualitative plot F01 Session1 0002](../../../outputs/personal_vad/qualitative_plots_live_ready/F01_Session1_0002.png)

In [8]:
%%bash
ls -lh ../../../outputs/personal_vad/qualitative_plots_live_ready/

total 1544
-rw-r--r--@ 1 harrydai  staff   228K Mar 25 14:30 F01_Session1_0001.png
-rw-r--r--@ 1 har

rydai  staff   236K Mar 25 14:30 F01_Session1_0002.png
-rw-r--r--@ 1 harrydai  staff   292K Mar 25 1

4:30 F01_Session1_0005.png
-rw-r--r--@ 1 harrydai  staff   946B Mar 25 14:30 summary.json
-rw-r--r--

@ 1 harrydai  staff   1.5K Mar 25 14:30 summary.txt


## Optional Extended Analysis

Run validation/test model selection and grouped error analysis when the required input evidence is available.

In [9]:
%%bash
../../../.venv/bin/python ../../../milestones/personal_vad/scripts/run_model_selection.py \
  --input ../../../outputs/personal_vad/ground_truth_eval/results.csv \
  --output-dir ../../../outputs/personal_vad/model_selection

../../../.venv/bin/python ../../../milestones/personal_vad/scripts/error_analysis.py \
  --input ../../../outputs/personal_vad/ground_truth_eval/results.csv \
  --output-dir ../../../outputs/personal_vad/error_analysis

cat ../../../outputs/personal_vad/model_selection/summary.txt
cat ../../../outputs/personal_vad/error_analysis/error_summary.txt

Wrote: ../../../outputs/personal_vad/model_selection/candidate_results.csv
Wrote: ../../../outputs/p

ersonal_vad/model_selection/summary.json
Wrote: ../../../outputs/personal_vad/model_selection/summar

y.txt
Wrote: ../../../outputs/personal_vad/model_selection/candidate_comparison.png


Wrote: ../../../outputs/personal_vad/error_analysis/grouped_metrics.csv
Wrote: ../../../outputs/pers

onal_vad/error_analysis/error_summary.json
Wrote: ../../../outputs/personal_vad/error_analysis/error

...

ssion1_0018 (0.6716)
energy_default: best=F01_Session1_0023 (0.8324), worst=F01_Session1_0005 (0.000

0), typical=F01_Session1_0004 (0.2010)

LIMITATIONS / CONFOUNDERS
----------------------------------

------------------------------------
- PHN-derived labels can disagree with perceived VAD boundaries

.
- Missing labels for some speakers can skew grouped comparisons.
- Metrics are aggregated and may 

hide timing-level boundary misses.


## Limitations

- PHN labels are best-available reference, not a perfect VAD oracle.
- Coverage can vary by speaker/fold, so claims are coverage-gated.
- Proxy tuning supports comparison and visualization but is not final generalization evidence.
- Optional analyses may be blocked when required validation evidence is missing.

## Summary

This workflow provides reproducible evidence for TinyVAD vs energy-baseline behavior through coverage auditing, conditional ground-truth evaluation, metric summaries, and qualitative inspection. The next improvement is broader labeled coverage across folds for stronger validation-driven selection.